# Notebook 2 – Strategy Backtests
> Educational only. Past performance ≠ future results.

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd, matplotlib.pyplot as plt
from data.fetcher import fetch_yfinance
from src.strategies import MeanReversionStrategy, MomentumStrategy
from src.backtester import VectorizedBacktester
from src.utils import plot_equity_curve, plot_signals
plt.style.use("seaborn-v0_8-darkgrid")

## 1. Mean Reversion Strategy (RSI + Bollinger Bands)

In [ ]:
df = fetch_yfinance("AAPL", start="2018-01-01", end="2024-01-01")
strategy = MeanReversionStrategy(lookback=20, rsi_period=14, rsi_oversold=30, rsi_overbought=70)
signals = strategy.generate_signals(df)
bt = VectorizedBacktester(initial_capital=100_000, commission=0.001, slippage=0.0005)
metrics = bt.run(signals)
print(metrics.summary())

In [ ]:
# Plot signals on price chart
plot_signals(signals, title="AAPL Mean Reversion Signals")
plt.show()

In [ ]:
# Buy-and-hold benchmark
bah = 100_000 * (1 + df["Returns"].fillna(0)).cumprod()
plot_equity_curve(metrics.equity_curve, benchmark=bah, title="Mean Reversion vs Buy-and-Hold")
plt.show()

## 2. Momentum Strategy (MACD + ADX)

In [ ]:
df2 = fetch_yfinance("QQQ", start="2018-01-01", end="2024-01-01")
mom = MomentumStrategy(fast_period=12, slow_period=26, adx_threshold=25, allow_short=False)
sig2 = mom.generate_signals(df2)
met2 = bt.run(sig2)
print(met2.summary())

In [ ]:
bah2 = 100_000 * (1 + df2["Returns"].fillna(0)).cumprod()
plot_equity_curve(met2.equity_curve, benchmark=bah2, title="Momentum (QQQ) vs Buy-and-Hold")
plt.show()

## 3. Pairs Trading (AAPL vs MSFT)

In [ ]:
from src.strategies import PairsTradingStrategy
df_a = fetch_yfinance("AAPL", start="2018-01-01", end="2024-01-01")
df_b = fetch_yfinance("MSFT", start="2018-01-01", end="2024-01-01")
pairs = PairsTradingStrategy(entry_z=2.0, exit_z=0.5, lookback=60)
# Monkey-patch to call the two-arg version
pair_signals = pairs.generate_signals.__func__(pairs, df_a, df_b)  # direct call
print(pair_signals[["SpreadZScore","Position_A","Position_B"]].tail(10))